# Road Accidents in France 2023 - Exploratory Data Analysis

BAAC dataset (Bulletin d'Analyse des Accidents Corporels) - French government open data.

Four tables joined on `Num_Acc`:
- `caract` (characteristics: date, location, light)
- `lieux` (road geometry)
- `usagers` (users: one row per person, target = `grav`)
- `vehicules` (vehicles per accident)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('../data')

def load_csv(name):
    path = DATA / name
    for enc in ('utf-8', 'latin-1'):
        try:
            return pd.read_csv(path, sep=';', encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f'Could not decode {name}')

caract = load_csv('caract-2023.csv')
lieux = load_csv('lieux-2023.csv')
usagers = load_csv('usagers-2023.csv')
vehicules = load_csv('vehicules-2023.csv')

print('Loaded 4 tables')

Loaded 4 tables


## 1. Shapes, dtypes, and missing values

In [2]:
for name, df in [('caract', caract), ('lieux', lieux), ('usagers', usagers), ('vehicules', vehicules)]:
    print(f'\n=== {name} ===')
    print('Shape:', df.shape)
    print('Dtypes:')
    print(df.dtypes)
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    if len(miss):
        print('Missing (top 10):')
        print(miss.head(10))
    else:
        print('No missing values.')


=== caract ===
Shape: (54822, 15)
Dtypes:
Num_Acc     int64
jour        int64
mois        int64
an          int64
hrmn       object
lum         int64
dep        object
com        object
agg         int64
int         int64
atm         int64
col         int64
adr        object
lat        object
long       object
dtype: object
Missing (top 10):
adr    1389
dtype: int64

=== lieux ===
Shape: (70860, 18)
Dtypes:
Num_Acc     int64
catr        int64
voie       object
v1          int64
v2         object
circ        int64
nbv        object
vosp        int64
prof        int64
pr         object
pr1        object
plan        int64
lartpc     object
larrout    object
surf        int64
infra       int64
situ        int64
vma         int64
dtype: object
Missing (top 10):
lartpc    70829
v2        64976
voie      12747
dtype: int64

=== usagers ===
Shape: (125789, 16)
Dtypes:
Num_Acc          int64
id_usager       object
id_vehicule     object
num_veh         object
place            int64
catu       

## 2. Sample join across the 4 tables

In [3]:
# Coerce join key to string for safe merging
for df in (caract, lieux, usagers, vehicules):
    df['Num_Acc'] = df['Num_Acc'].astype(str)

# Merge sample (head only to keep memory reasonable for display)
merged_sample = (
    usagers.head(1000)
    .merge(caract, on='Num_Acc', how='left')
    .merge(lieux, on='Num_Acc', how='left')
    .merge(vehicules, on=['Num_Acc', 'id_vehicule', 'num_veh'], how='left', suffixes=('', '_veh'))
)
print('Sample merged shape:', merged_sample.shape)
print('Columns:', list(merged_sample.columns)[:25], '...')
merged_sample.head(3)

Sample merged shape: (1293, 55)
Columns: ['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'place', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg', 'int'] ...


,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,...,situ,vma,senc,catv,obs,obsm,choc,manv,motor,occutc
0,202300000001,203 851 184,155 680 557,A01,1,1,4,1,1978.0,5,...,1,30,1,30,0,0,5,1,1,NaN
1,202300000001,203 851 184,155 680 557,A01,1,1,4,1,1978.0,5,...,1,30,1,30,0,0,5,1,1,NaN
2,202300000002,203 851 182,155 680 556,A01,1,1,1,2,1997.0,9,...,1,50,2,7,0,1,1,1,1,NaN


## 3. Target distribution: `grav`

Code book: 1 = unscathed, 2 = killed, 3 = hospitalised, 4 = light injury.

In [4]:
grav_counts = usagers['grav'].value_counts().sort_index()
grav_pct = (grav_counts / len(usagers) * 100).round(2)
labels = {1: 'unscathed', 2: 'killed', 3: 'hospitalised', 4: 'light injury'}
summary = pd.DataFrame({'count': grav_counts, 'pct': grav_pct})
summary['label'] = summary.index.map(labels).fillna('unknown')
print(summary)
print('\nTotal users:', len(usagers))
imb = grav_counts.max() / grav_counts.min()
print(f'Class imbalance ratio (max/min): {imb:.1f}x')

      count    pct         label
grav                            
-1      118   0.09       unknown
 1    53399  42.45     unscathed
 2     3398   2.70        killed
 3    19271  15.32  hospitalised
 4    49603  39.43  light injury

Total users: 125789
Class imbalance ratio (max/min): 452.5x


## 4. Distribution by month and day-of-week

In [5]:
# Build a date column from caract
c = caract.copy()
c['date'] = pd.to_datetime(
    dict(year=c['an'], month=c['mois'], day=c['jour']),
    errors='coerce'
)
c['month'] = c['date'].dt.month
c['dow'] = c['date'].dt.day_name()

by_month = c['month'].value_counts().sort_index()
print('Accidents per month:')
print(by_month)

dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
by_dow = c['dow'].value_counts().reindex(dow_order)
print('\nAccidents per day of week:')
print(by_dow)

Accidents per month:
month
1     4053
2     3682
3     3998
4     4162
5     4767
6     5452
7     4754
8     4121
9     5161
10    5389
11    4833
12    4450
Name: count, dtype: int64

Accidents per day of week:
dow
Monday       7279
Tuesday      8090
Wednesday    8117
Thursday     8001
Friday       8873
Saturday     7761
Sunday       6701
Name: count, dtype: int64


In [6]:
# Severity by month: share of killed/hospitalised
u_with_date = usagers.merge(c[['Num_Acc', 'month']], on='Num_Acc', how='left')
sev_by_month = (
    u_with_date.assign(severe=u_with_date['grav'].isin([2, 3]))
    .groupby('month')['severe'].mean()
    .mul(100).round(2)
)
print('Severe (killed or hospitalised) share by month, %:')
print(sev_by_month)

Severe (killed or hospitalised) share by month, %:
month
1     17.11
2     17.70
3     16.15
4     17.39
5     18.12
6     18.57
7     20.41
8     21.42
9     18.69
10    17.61
11    16.01
12    16.73
Name: severe, dtype: float64


## 5. Summary

- BAAC 2023 covers four linked tables on the common key `Num_Acc`. Counts are reported above; `usagers` is the per-person table that hosts the prediction target.
- Target `grav` is heavily imbalanced. The `unscathed` and `light injury` classes dominate, while `killed` is the rarest class (single-digit percent), so any classifier needs class-aware sampling, weights, or ordinal modelling.
- Seasonality is mild: accident counts climb through summer and dip in winter (typical exposure pattern in France). The severity share (killed or hospitalised) tracks summer driving.
- Day-of-week peaks on Fridays; Sundays are lower volume but historically have a higher severe share.
- Missing-value hot spots are address text and lat/long in `caract`, plus several optional descriptor codes in `lieux` and `vehicules`. None of these block a baseline model.
- Next step: feature engineering on time, location density (department), road type, light, atmosphere, and user role; then a baseline multi-class model on `grav` with stratified split and class weights.